# Codificacion Variables Categorigas, Angulares y Temporales con metodo por Transecto y metodo General

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Codificación de variables: cíclicas, direccionales, one-hot (ML) y factorize (DL).
Se aplica a transectos y a estaciones individuales.
"""

import os
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path

BASE_DIR = os.path.expanduser("/Volumes/copia seguridad1/enviar_benja/carpeta sin título/clean/Finales/")
ENCODED_DIR = os.path.join(BASE_DIR, "encoded")

INPUT_BY_TRANSECT = os.path.join(BASE_DIR, "imputed_by_transect")
INPUT_GLOBAL = os.path.join(BASE_DIR, "imputed_global")

OUTPUT_ML_TRANSECT = os.path.join(ENCODED_DIR, "ml", "by_transect")
OUTPUT_DL_TRANSECT = os.path.join(ENCODED_DIR, "dl", "by_transect")
OUTPUT_ML_GLOBAL = os.path.join(ENCODED_DIR, "ml", "global")
OUTPUT_DL_GLOBAL = os.path.join(ENCODED_DIR, "dl", "global")

for path in [OUTPUT_ML_TRANSECT, OUTPUT_DL_TRANSECT, OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL]:
    os.makedirs(path, exist_ok=True)

NUM_COLS = ['NO', 'NO2', 'NOx', 'O3', 'Veloc.', 'Direc.', 'Temp.', 'R.Sol.', 'Dist.', 'Angulo']
CATEGORICAL_COLS = ['Estacion', 'Transecto']


def cyclical_encode(series, period):
    rad = 2 * np.pi * series / period
    return np.sin(rad), np.cos(rad)


def standardize_target_column(df):
    df = df.copy()
    if "O3_for_impute" in df.columns and "O3" not in df.columns:
        df.rename(columns={"O3_for_impute": "O3"}, inplace=True)
    elif "O3_for_impute" in df.columns and "O3" in df.columns:
        df["O3"] = df["O3"].where(df["O3"].notna(), df["O3_for_impute"])
        df.drop(columns=["O3_for_impute"], inplace=True)
    return df


def add_datetime_features(df):
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("El índice debe ser DatetimeIndex")
    idx = df.index
    hour = idx.hour
    dayofyear = idx.dayofyear
    week = idx.isocalendar().week.astype(int)
    month = idx.month
    year = idx.year
    df['hour_sin'], df['hour_cos'] = cyclical_encode(hour, 24)
    df['day_sin'], df['day_cos'] = cyclical_encode(dayofyear, 365)
    df['week_sin'], df['week_cos'] = cyclical_encode(week, 52)
    df['month_sin'], df['month_cos'] = cyclical_encode(month, 12)
    df['year'] = year
    return df


def add_directional_features(df):
    df = df.copy()
    for col in ['Direc.', 'Angulo']:
        if col in df.columns:
            values = pd.to_numeric(df[col], errors="coerce")
            rad = np.radians(values)
            df[f'{col}sin'] = np.sin(rad)
            df[f'{col}cos'] = np.cos(rad)
            df.drop(columns=[col], inplace=True)
    return df


def normalize_station(val):
    s = str(val).strip()
    m = re.search(r'Estacion[ _]?(\d+)', s, re.IGNORECASE) or re.search(r'\bE[ _]?(\d+)\b', s, re.IGNORECASE) or re.search(r'(\d+)', s)
    return f"Estacion_{m.group(1)}" if m else s.replace(' ', '_')


def normalize_transect(val):
    s = str(val).strip()
    m = re.search(r'Transecto[ _]?(\d+)', s, re.IGNORECASE) or re.search(r'\bT[ _]?(\d+)\b', s, re.IGNORECASE) or re.search(r'(\d+)', s)
    return f"Transecto_{m.group(1)}" if m else s.replace(' ', '_')


def encode_categorical_ml(df, cat_cols):
    df = df.copy()
    existing = [c for c in cat_cols if c in df.columns]
    if not existing:
        return df
    if 'Estacion' in existing:
        df['Estacion'] = df['Estacion'].apply(normalize_station)
    if 'Transecto' in existing:
        df['Transecto'] = df['Transecto'].apply(normalize_transect)
    dummies = pd.get_dummies(df[existing].astype(str), prefix='', prefix_sep='')
    dummies.columns = [col.replace(' ', '_').replace('/', '_') for col in dummies.columns]
    df = pd.concat([df, dummies], axis=1)
    df.drop(columns=existing, inplace=True)
    return df


def encode_categorical_dl(df, cat_cols, save_mapping=True, mapping_file=None):
    df = df.copy()
    existing = [c for c in cat_cols if c in df.columns]
    if not existing:
        return df, {}
    mapping = {}
    for col in existing:
        codes, uniques = pd.factorize(df[col], sort=False)
        df[col] = codes
        mapping[col] = {str(category): int(code) for code, category in enumerate(uniques)}
    if save_mapping and mapping_file:
        with open(mapping_file, 'w', encoding='utf-8') as f:
            json.dump(mapping, f, indent=2)
    return df, mapping


def prepare_dataframe(df):
    df = df.copy()
    df = standardize_target_column(df)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, errors="coerce")
    df = df.loc[~df.index.isna()].copy()
    df = df.sort_index(kind="mergesort")
    return df


def process_file(input_path, output_ml_dir, output_dl_dir, is_global=False):
    base_name = Path(input_path).stem
    print(f"Procesando: {base_name} (global={is_global})")
    df = pd.read_csv(input_path, index_col=0, parse_dates=True, low_memory=False)
    df = prepare_dataframe(df)
    df = add_datetime_features(df)
    df = add_directional_features(df)
    # ML: one-hot
    df_ml = encode_categorical_ml(df.copy(), CATEGORICAL_COLS)
    # DL: factorize
    mapping_file = os.path.join(output_dl_dir, f"{base_name}_mapping.json")
    df_dl, _ = encode_categorical_dl(df.copy(), CATEGORICAL_COLS, save_mapping=True, mapping_file=mapping_file)
    df_ml.to_csv(os.path.join(output_ml_dir, f"{base_name}.csv"), index=True)
    df_dl.to_csv(os.path.join(output_dl_dir, f"{base_name}.csv"), index=True)
    print(f"  ML guardado, DL guardado")


def process_by_transect():
    if not os.path.exists(INPUT_BY_TRANSECT):
        print(f"La carpeta {INPUT_BY_TRANSECT} no existe.")
        return
    files = list(Path(INPUT_BY_TRANSECT).glob("*.csv"))
    for f in files:
        process_file(f, OUTPUT_ML_TRANSECT, OUTPUT_DL_TRANSECT, is_global=False)


def process_global():
    if not os.path.exists(INPUT_GLOBAL):
        print(f"La carpeta {INPUT_GLOBAL} no existe.")
        return
    files = list(Path(INPUT_GLOBAL).glob("*.csv"))
    for f in files:
        process_file(f, OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL, is_global=True)


if __name__ == "__main__":
    print("Iniciando codificación de variables...")
    process_by_transect()
    process_global()
    print("Proceso completado.")

Iniciando codificación de variables...
Procesando: Transecto_1 (global=False)
  ML guardado, DL guardado
Procesando: Transecto_2 (global=False)
  ML guardado, DL guardado
Procesando: Transecto_3 (global=False)
  ML guardado, DL guardado
Procesando: Transecto_4 (global=False)
  ML guardado, DL guardado
Procesando: Transecto_6 (global=False)
  ML guardado, DL guardado
Procesando: Transecto_7 (global=False)
  ML guardado, DL guardado
Procesando: Transecto_8 (global=False)
  ML guardado, DL guardado
Procesando: T1_E1_Alicante (global=True)
  ML guardado, DL guardado
Procesando: T1_E2_Elda (global=True)
  ML guardado, DL guardado
Procesando: T2_E1_Elche (global=True)
  ML guardado, DL guardado
Procesando: T2_E2_Elda (global=True)
  ML guardado, DL guardado
Procesando: T3_E1_Valencia (global=True)
  ML guardado, DL guardado
Procesando: T3_E2_Buñol (global=True)
  ML guardado, DL guardado
Procesando: T4_E1_Valencia (global=True)
  ML guardado, DL guardado
Procesando: T4_E2_Villar_Arzobispo (